### 2.1 连续动作问题精讲：DQN 为什么"卡在"这里（🔴 0 基础也能懂）

**本节导读**：做连续控制前，先精确回答"为什么 DQN 不行"。答案不是
"动作多"，而是 **max 枚举在数学上不可行、离散化在维度上不可行**。

**1. 问题设定**：动作 $a\in\mathbb{R}^d$（机器人关节力矩、油门刹车）。
DQN 的更新要 $\max_{a'} Q(s',a')$——遍历所有动作找最大 Q。

**2. 为什么枚举不了（推导级理解）**：
- 离散动作 Q 表：输出层 |A| 个节点，argmax = 一次前向
- 连续动作：$a'$ 是**连续向量**，穷举无限个点；即使网格化，
  维度 $d$ 每维切 $N$ 份 → $N^d$ 个点（维度灾难，00 篇 §19）
- 对每个 (s, a) 逐点前向的代价：$O(N^d)$ 次前向 → 不可行

**3. 三个"假装能行"的错误方案（面试防踩坑）**：
- ❌ "随机采样几个动作取 max"：近似太粗，且梯度无法穿过 argmax
- ❌ "把动作离散成很细的网格"：维度爆炸（$N^d$）
- ❌ "Q 网络输入 (s,a) 输出标量，再对 a 梯度上升"：Q 非凸，找不到全局 max
- ✅ 正确思路：**再训练一个 Actor 网络"直接输出最优动作"**——让 max
  从"枚举"变成"学一个函数"，这正是 DPG/DDPG 的出发点

**4. 一句话框架**：DPG 定理说
$$\max_a Q(s,a) \;\text{等价于学}\; \mu_\theta(s)\approx\arg\max_a Q(s,a)$$
用 Actor 把"求 max"替换成"网络前向"——**枚举问题变成函数拟合问题**。

**5. 易错点**：
- "连续动作"还包括**高维离散**（如词表太大）→ 同样用策略梯度系
- 连续动作的 **Actor 输出是动作本身**（不是动作分布；随机策略见 SAC）

**6. 面试问法**："DQN 为什么不能做连续控制？" → "max 需要枚举动作，
连续/高维下不可行；对策是 Actor 网络直接输出动作（DPG）或高斯策略
（SAC）学分布。"

---


# 🎮 08 · 连续动作控制：DDPG / TD3 / SAC（从方向盘到机械臂）

> 面试定位：**连续动作**（Continuous Control）是机器人、自动驾驶、工业控制的标配，
> DDPG/TD3/SAC 三件套是算法岗必考；SAC 更是现代 RL 的"默认选项"。
> 知识链：04 的 DQN 处理**离散动作**（枚举 argmax），本篇解决"动作是连续数值"怎么办；
> 05 的 Actor-Critic 在此升级为 **确定性 Actor + 连续 Critic** 的四网络结构。
> 标记：🔴 必背 / 🟡 掌握 / 🟢 了解。

---

## 1. 先讲清楚问题：什么是连续动作（🔴 0 基础也能懂）

### 通俗例子

- **离散动作**：雅达利游戏的"上/下/左/右"、老虎机的"摇/不摇"——动作是**有限的几个选项**，DQN 能枚举。
- **连续动作**：汽车的**方向盘转角**（-540°~540° 任意值）、机械臂**每个关节的力矩**、飞行器的油门——
  动作是一个**连续数值**，可能是 3.1415926……这种"无穷多个选项"。

> 一句话记忆：**离散动作 = 选菜单；连续动作 = 拧旋钮。** 旋钮能拧到任意角度，
> 不能靠"枚举每个角度"来选最好的。

### 为什么 DQN 直接失效（🔴）

DQN 的决策是 $a = \arg\max_a Q(s,a)$ —— 它需要一个**能遍历所有动作的 argmax**。
- 离散 18 个动作：遍历一遍，轻松。
- 连续动作：$a \in \mathbb{R}^k$（$k$ 个关节），遍历无穷个点，不可能。

**方案**：既然不能"遍历动作选 Q 最大"，那就让**另一个网络直接生成动作**——
这就是 **Actor（演员）**：$a = \mu_\theta(s)$。演员负责"决定动作"，评论家负责"评价动作"。

---

## 2. 确定性策略梯度定理（DPG）（🔴 核心推导）

### 目标函数与链式法则

策略 $\mu_\theta$ 是**确定性**的（给定状态直接输出动作，不带随机采样）。
目标：$J(\theta) = \mathbb{E}_{s \sim \rho^\mu}\big[Q^{\mu}(s, \mu_\theta(s))\big]$

对 $\theta$ 求梯度（**确定性策略梯度定理**，Silver et al. 2014）：

$$\nabla_\theta J = \mathbb{E}_{s}\Big[\underbrace{\nabla_a Q^{\mu}(s, a)\big|_{a=\mu_\theta(s)}}_{\text{评论家告诉：动作往哪挪 Q 变大}} \cdot \underbrace{\nabla_\theta \mu_\theta(s)}_{\text{演员告诉：参数怎么动动作才这么挪}}\Big]$$

### 四步直觉推导（面试手推）

1. $J(\theta) = \mathbb{E}_s[Q(s, \mu_\theta(s))]$：先固定状态，J 通过 $Q$ 依赖 $\theta$
2. 链式法则：$\nabla_\theta J = \mathbb{E}_s\Big[\frac{\partial Q}{\partial a}\frac{\partial a}{\partial \theta}\Big]$
3. $\frac{\partial a}{\partial \theta}$ 就是 $\nabla_\theta \mu_\theta(s)$（演员网络的反向传播）
4. $\frac{\partial Q}{\partial a}$ 由评论家网络对**输入动作**求导给出（Q 网络可微）

> 与随机策略梯度对比：随机版是 $\nabla_\theta J = \mathbb{E}[\nabla_\theta \log \pi_\theta \cdot A]$（05 篇），
> 要采样动作；确定性版**不用对动作采样**，直接沿 Q 的梯度方向推演员 —— 方差更小，但需要 Q 可微。

## 核心概念

### (a) 🗂️ 连续动作控制（DDPG / TD3 / SAC）知识地图

这篇解决"连续动作"问题：DQN 只能枚举离散动作，而方向盘、机械臂需要连续值。主角是 DDPG（DQN 的连续化四网络）、TD3（对 DDPG 的三刀手术）和 SAC（熵正则化的最大熵方法）。

```mermaid
flowchart TB
    Node1["🧠 连续动作问题"]
    Node1 --> Node2["🧠 确定性策略梯度 DPG"]
    Node2 --> Node3["🧠 DDPG 四网络"]
    Node3 --> Node4["🧠 软更新 Polyak"]
    Node3 --> Node5["🧠 OU 噪声探索"]
    Node3 --> Node6["🧠 TD3 三大手术"]
    Node6 --> Node7["🧠 双 Q 取 min"]
    Node3 --> Node8["🧠 SAC 最大熵"]
    Node8 --> Node9["🧠 温度 α 自适应与选型"]
```

- **节点1（连续动作问题）**：动作是连续值（如转向角 -1~1）时无法枚举出 max，DQN 的 argmax 直接失效；需要用"输出一个动作值"的策略网络加上"对动作可微"的 Q 网络。
- **节点2（确定性策略梯度 DPG）**：对确定性策略 μ(s)，梯度可以写成 ∇θJ = E[∇a Q · ∇θ μ]，用链式法则把"动作空间上的梯度"变成"参数空间上的梯度"，是 DDPG 的数学地基。
- **节点3（DDPG 四网络）**：Actor 在线/目标 + Critic 在线/目标 共四个网络：Actor 输出动作，Critic 评价 (s,a)，目标网络算稳定目标，分工像"运动员 + 裁判 + 两个记分员"。
- **节点4（软更新 Polyak）**：DDPG 不用"每 C 步硬拷贝"，而是每步按 τ 做指数滑动平均 θ_target ← τθ + (1−τ)θ_target，目标网络"慢半拍"地跟随，训练更平滑。
- **节点5（OU 噪声探索）**：确定性策略不自己探索，需要往动作上叠加 OU 噪声（时序相关、更平滑）或高斯噪声，逼它多试新动作。
- **节点6（TD3 三大手术）**：① 双 Q 取 min 压低高估、② 目标策略平滑（对目标动作加噪声）防 Q 函数过尖、③ 延迟更新（Critic 多更几次再更 Actor），三刀各有其必要。
- **节点7（双 Q 取 min）**：用两个 Critic 估计取较小值当目标，用"低估换稳定"抵消 Q 的过估，是 TD3 最重要的一刀。
- **节点8（SAC 最大熵）**：SAC 把目标从"最大回报"改成"最大回报 + 熵"，主动鼓励随机、探索更充分，动作分布有闭式解（玻尔兹曼形式），是现代连续控制的默认选择。
- **节点9（温度 α 自适应与选型）**：α 用对偶梯度自动调节（熵太低就升 α 更随机）；选型口诀：简单任务 DDPG，怕高估上 TD3，要稳要探索选 SAC。

In [1]:
# ---- matplotlib 中文字体（防图内中文乱码）----
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Songti SC', 'Arial Unicode MS', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
# -------------------------------------------
# ========== DPG 定理的数值验证：解析梯度 vs 中心差分 ==========
# 玩具设定：Q(s,a) = 0.5·aᵀWᵀs（二次型，关于 a 可微）；确定性策略 μθ(s) = sᵀθ
import numpy as np
rng = np.random.default_rng(0)
d, k = 3, 2                      # 状态 3 维，动作 2 维
W = rng.normal(size=(d, k))
theta = rng.normal(size=(d, k))
s = rng.normal(size=d)

def J(th):                        # 目标函数：Q(s, μθ(s))
    mu = s @ th
    return 0.5 * float(mu @ (W.T @ s))

# 解析梯度：dJ/dθ = ∇aQ · dμ/dθ = (0.5·Wᵀs) 外积 s
g_ana = np.outer(s, 0.5 * (W.T @ s))

# 数值梯度：对每个参数扰动 ±ε，中心差分
eps = 1e-6
g_num = np.zeros((d, k))
for i in range(d):
    for j in range(k):
        t1 = theta.copy(); t1[i, j] += eps
        t2 = theta.copy(); t2[i, j] -= eps
        g_num[i, j] = (J(t1) - J(t2)) / (2 * eps)

err = np.abs(g_ana - g_num).max()
print('DPG 解析梯度 vs 中心差分 最大误差: %.2e' % err)
assert err < 1e-6
print('定理验证 ✓：∇θJ = ∇aQ·∇θμ —— 演员沿「Q 上升最快的动作方向」更新')
print('（随机策略梯度要采样动作估期望，DPG 直接求导 → 方差小，但要求 Q 网络可微）')

DPG 解析梯度 vs 中心差分 最大误差: 1.50e-11
定理验证 ✓：∇θJ = ∇aQ·∇θμ —— 演员沿「Q 上升最快的动作方向」更新
（随机策略梯度要采样动作估期望，DPG 直接求导 → 方差小，但要求 Q 网络可微）


### 为什么 DDPG 需要"四个网络"——一张图画清楚（🔴 深入）

<img src="images/08_robot_standup.png" width="560">

上图是连续控制（机器人起身）的真实场景：**动作空间是连续力矩**，DQN 的
argmax 枚举做不到，DDPG 换成了"Actor 直接输出动作 + Critic 评价 (s,a)"。
但为了稳定，它把两组网络各复制一份 → **四个网络**：

| 网络 | 干什么 | 为什么需要 |
|---|---|---|
| Actor $\mu_\theta$ | 在线输出动作 | 连续动作的决策者 |
| Critic $Q_\omega$ | 在线评价 (s,a) | 提供 DPG 梯度的"方向" |
| 目标 Actor $\mu_{\theta^-}$ | 算目标动作 | 目标 $y$ 里的 $a'$ 必须"慢变" |
| 目标 Critic $Q_{\omega^-}$ | 算目标价值 | 目标 $y$ 里的 $Q$ 必须"慢变" |

**软更新的作用（关键）**：每步都把目标网络往在线网络**挪一小步**：
$$\theta^- \leftarrow \tau\theta + (1-\tau)\theta^-,\quad \tau=0.005$$
- τ 大 → 目标追得急 → 又变成"自举自己"，不稳
- τ 小 → 目标像"慢半拍的回声" → 训练稳定（本篇 code cell 演示了跟随曲线）

> 一句话：**四个网络 = 决策/评价各一对（在线 + 慢速副本）；目标是"冻结的锚"，**
> **软更新让锚缓慢移动，既跟得上学习又不震荡。**

---


### 3.1 DPG 定理精讲：确定性策略的梯度长什么样（🔴 核心推导）

**本节导读**：随机策略梯度 $\nabla J=\mathbb{E}[\nabla\log\pi\,A]$ 在
连续动作**方差大、样本贵**。确定性策略用"动作=函数"，梯度里不再有
log π 的期望。这一节把 DPG 定理推出来（Silver et al. 2014）。

**1. 确定性策略**：$\mu_\theta(s)$ 直接输出动作（不采样）。目标
$$J(\theta) = \int_{\mathcal{S}} d^{\mu_\theta}(s)\, Q^{\mu_\theta}\big(s, \mu_\theta(s)\big)\, ds$$

**2. 确定性策略梯度定理（背公式）**：
$$\nabla_\theta J(\theta) = \mathbb{E}_{s\sim d^{\mu_\theta}}\Big[
\underbrace{\nabla_a Q^{\mu_\theta}(s,a)\big|_{a=\mu_\theta(s)}}_{\text{① 动作方向}} \cdot
\underbrace{\nabla_\theta \mu_\theta(s)}_{\text{② 参数方向}}\Big]$$
- ① Q 对动作的梯度：告诉我们"往哪个方向调动作，Q 升最快"
- ② 策略对参数的梯度：告诉我们"参数怎么动，动作就往那个方向走"
- 二者点乘 = "沿 Q 上升方向调整策略"——**链式法则一步到位**

**3. 对比随机 PG（推导级差异）**：
- 随机版：$\mathbb{E}[\nabla\log\pi(a|s)\,A]$ 要在动作空间采样平均
- 确定性版：不再对动作采样（只对状态平均），**方差小、样本高效**
- 代价：确定性策略**不天然探索** → 必须外加噪声（DDPG 的 OU 噪声）

**4. 为什么"链式"合法（关键假设）**：$\mu_\theta$ 对 θ 可微、
$Q$ 对 a 可微 → $\nabla_\theta Q(s,\mu_\theta(s))$ 用链式法则分解。
离策略也成立（行为策略可以用带噪声版本：$\beta=\mu+\text{噪声}$）→
**DDPG 因此天然 off-policy**（回放合法）。

**5. 易错点**：
- 梯度里是 $\nabla_a Q$（**对动作**求导，不是对参数），方向由 Critic 给
- 确定性策略 ≠ 最优策略：只是"动作确定"，可能陷局部（噪声救）

**6. 面试问法**："DPG 定理是什么？" → "∇J=E[∇_a Q·∇_θ μ]，链式法则
把'调 Q 最高的动作方向'与'参数到动作的方向'连起来；不再对动作采样，
方差小，off-policy 合法。"

---


## 3. DDPG：DQN 的连续化四网络结构（🔴）

**DDPG（Deep Deterministic Policy Gradient，Lillicrap 2016）** = DPG 定理 + DQN 的经验回放/目标网络。

### 四个网络各司其职

| 网络 | 做什么 | 谁在更新 | 更新方式 |
|---|---|---|---|
| **Actor**（在线） | 输入状态 → 输出动作 $a=\mu_\theta(s)$ | 每步 | 沿 DPG 梯度上升 |
| **Critic**（在线） | 输入 (状态, 动作) → 输出 $Q_\phi(s,a)$ | 每步 | TD 误差：$\mathcal{L} = \mathbb{E}[(y - Q_\phi)^2]$ |
| **Target Actor** | 生成目标动作 $\mu_{\theta'}(s')$ | 软更新 | $\theta' \leftarrow \tau\theta + (1-\tau)\theta'$ |
| **Target Critic** | 生成目标值 $Q_{\phi'}(s', a')$ | 软更新 | $\phi' \leftarrow \tau\phi + (1-\tau)\phi'$ |

### 目标值（与 DQN 的差别）

$$y = r + \gamma\, Q_{\phi'}\big(s', \mu_{\theta'}(s')\big)$$

注意：DQN 的目标用 $\max_a Q(s',a)$（枚举），DDPG 用**目标演员**直接给出 $a'$ —— 这就是"连续版的 max"。

### 软更新（Polyak averaging）为什么稳

目标网络不"大步拷贝"在线网络，而是每次只挪 $\tau$（通常 0.005）：
$$\theta' \leftarrow \tau\theta + (1-\tau)\theta'$$
→ 目标值变化慢 → TD 误差的"靶子"稳定 → 训练不发散。

In [2]:
# ========== 软更新演示：目标网络"慢半拍"地跟随在线网络 ==========
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
rng = np.random.default_rng(1)
tau = 0.005
T = 400
theta, theta_t = 0.0, 0.0
thetas, targets = [], []
for _ in range(T):
    theta += 0.05 * (1.5 - theta)              # 在线参数朝 1.5 学（模拟训练）
    theta_t = tau * theta + (1 - tau) * theta_t  # 目标参数软更新
    thetas.append(theta); targets.append(theta_t)
fig, ax = plt.subplots(figsize=(7.2, 4.2))
ax.plot(thetas, lw=1.8, color='#3182bd', label='在线网络 θ（每步都在变）')
ax.plot(targets, lw=1.8, color='#C44E52', label=f'目标网络 θ_t（τ={tau} 慢跟随）')
ax.set_xlabel('训练步数'); ax.set_ylabel('参数值')
ax.set_title('软更新：θ⁻ = τθ + (1−τ)θ⁻ —— 目标永远"慢半拍"，TD 靶子才稳定', fontsize=12)
ax.legend(fontsize=9); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
assert theta_t < theta                       # 目标滞后于在线
print(f'末尾：在线 θ={theta:.3f}，目标 θ⁻={theta_t:.3f}（滞后 → 目标值变化缓慢，训练稳定）')

末尾：在线 θ=1.500，目标 θ⁻=1.277（滞后 → 目标值变化缓慢，训练稳定）


C:\Users\24260\AppData\Local\Temp\ipykernel_34248\3708714971.py:19: UserWarning: Glyph 8315 (\N{SUPERSCRIPT MINUS}) missing from font(s) Microsoft YaHei.
  ax.legend(fontsize=9); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
C:\Users\24260\AppData\Local\Temp\ipykernel_34248\3708714971.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.legend(fontsize=9); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()


## 4. 连续动作怎么探索：OU 噪声（🟡）

离散动作靠 ε-greedy 随机选动作探索；连续动作怎么办？—— 给动作**加时间相关的噪声**。

**Ornstein-Uhlenbeck（OU）过程**（湍流-乌伦贝克过程）：

$$dx = \theta(\mu - x)\,dt + \sigma\,dW$$

通俗理解：**一根被拉离原点的橡皮筋**——每次随机扰动让它动，但 $\theta(\mu-x)$ 总把它拉回均值 $\mu$。
- 与独立高斯噪声的区别：OU 噪声**时间相关**（上一步偏左，下一步大概率还偏左）→ 动作平滑、不抖动
- 连续控制（机械臂/车辆）最怕动作抖，OU 是 DDPG 的经典探索噪声

In [3]:
# ========== OU 噪声 vs 独立高斯噪声：平滑性对比 ==========
def ou_noise(rng, T=300, theta=0.15, sigma=0.25, mu=0.0, x0=0.0):
    x = x0; xs = []
    for _ in range(T):
        x += theta * (mu - x) + sigma * rng.normal()
        xs.append(x)
    return np.array(xs)

rng = np.random.default_rng(3)
ou1 = ou_noise(rng, sigma=0.25)
ou2 = ou_noise(rng, sigma=0.5)
iid = rng.normal(0, 0.35, len(ou1))
fig, ax = plt.subplots(figsize=(7.6, 4.4))
ax.plot(ou1, lw=1.4, label='OU(σ=0.25)：平滑、缓慢来回')
ax.plot(ou2, lw=1.4, label='OU(σ=0.50)：更大范围探索')
ax.plot(iid, lw=1.0, alpha=0.7, label='独立高斯：每步独立、抖动大')
ax.set_xlabel('时间步'); ax.set_ylabel('噪声值')
ax.set_title('探索噪声：OU 时间相关（平滑）vs iid 白噪声（抖动）', fontsize=12)
ax.legend(fontsize=9); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
lag1 = np.corrcoef(ou1[:-1], ou1[1:])[0, 1]
lag1_iid = np.corrcoef(iid[:-1], iid[1:])[0, 1]
print('lag-1 自相关：OU=%.3f（相邻相关 → 动作不跳变） vs iid=%.3f（≈0）' % (lag1, lag1_iid))
assert lag1 > 0.3 and abs(lag1_iid) < 0.1

lag-1 自相关：OU=0.844（相邻相关 → 动作不跳变） vs iid=0.043（≈0）


C:\Users\24260\AppData\Local\Temp\ipykernel_34248\3712092943.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.legend(fontsize=9); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()


### 5.1 TD3 三大手术精讲：每一刀为什么必要（🔴 面试必问）

**本节导读**：DDPG 能跑但脆：Q 高估、critic 训练慢、策略更新有噪声。
TD3（2018）动三刀：**clipped double Q、延迟更新、目标平滑**。每一刀
都有明确的"病灶"对应。

**1. 第一刀：Clipped Double Q（治高估）**
两个 Critic 独立训练，目标取**较小**的那个：
$$y = r + \gamma\, \min_{i=1,2} Q_{\theta_i^-}\big(s', \mu_{\theta^-}(s')\big)$$
- 高估来自 max 偏爱噪声；两个独立估计噪声不同向，min 取"不虚高"的
- 结论：**对低估免疫**（智能体宁可保守）——比 Double Q 更稳

**2. 第二刀：延迟更新（治"过拟合的 Q"）**
- 病灶：Actor 每次更新都按"当前尽力拟合的 Q"走，Q 还差时 Actor 被带偏
- 手术：Actor（和目标网络）**每 2 步才更新 1 次**，Critic 多训几步
  → Actor 总在"更可信的 Q"上行进：$d_{\text{delay}}=2$
- 直觉：**让 Critic 先跑稳，再来指点 Actor**

**3. 第三刀：目标平滑（治"Q 的尖峰"）**
目标动作加小噪声 + 截断：
$$a' = \text{clip}\big(\mu_{\theta^-}(s') + \varepsilon,\ a_{\min}, a_{\max}\big),\quad \varepsilon\sim\mathcal{N}(0,\sigma)$$
- 病灶：确定性策略在"一个点"上取 max，Q 的尖峰（拟合误差）被精确踩中
- 手术：目标里平滑 → Q 学"邻域平均"，尖峰被磨平（正则化）；
  clip 保证噪声不越出动作边界（安全）
- σ 常取 0.2、噪声按 γ 缩放（远期间接影响）

**4. 三刀各自管什么（背表）**：
| 手术 | 病灶 | 一句话 |
|---|---|---|
| min 双 Q | 高估放大 | 取保守估计，低估免疫 |
| 延迟更新 | Actor被带偏 | 让 Critic 先准再指点 |
| 目标平滑 | Q 尖峰过拟合 | 邻域平均磨平噪声尖峰 |

**5. 易错点**：
- 三个目标网络（μ⁻,Q₁⁻,Q₂⁻）也要按延迟同步（软更新 τ 照旧）
- 平滑噪声**加了要 clip**：动作越界 → 价值突变 → 反而有害

**6. 面试问法**："TD3 对 DDPG 改了什么？" → "min 双 Q 治高估、延迟
更新治带偏、目标平滑治尖峰；三个手术合起来是连续控制的标配。"

---


## 5. TD3：对 DDPG 的三大手术（🔴 面试必问）

DDPG 的毛病：**Critic 会高估 Q 值**（与 DQN 的高估同源，04 篇已证 $E[\max] \ge \max E$），
连续动作下高估会被 Actor 放大（Actor 专挑被高估的动作）。

**TD3（Twin Delayed DDPG，Fujimoto 2018）三刀**：

1. **双 Q 取 min**：两个 Critic，目标取 $y = r + \gamma\,\min(Q_{\phi_1'}, Q_{\phi_2'})$ —— 有意低估
2. **延迟更新 Actor**：Actor 每 2 步才更新一次，让 Critic 先学准
3. **目标策略平滑**：目标动作加小噪声 $a' = \mu_{\theta'}(s') + \text{clip}(\mathcal{N}(0,\sigma),-c,c)$
   —— 让 Q 曲面"平滑"，避免钻尖峰

> 一句话：**TD3 = DDPG + 双 Q + 慢半拍的 Actor + 目标动作加噪**（三个"防高估/防钻尖"的小心机）。

In [4]:
# ========== 双 Q 取 min：低估换稳定（数值演示） ==========
rng = np.random.default_rng(4)
X, n = 1.0, 40000
e1 = rng.normal(0, 1.0, n); e2 = rng.normal(0, 1.0, n)   # 两个 Critic 对同一真值 X 的独立噪声估计
q1, q2 = X + e1, X + e2
m = np.minimum(q1, q2).mean()
print('E[min(Q1,Q2)] = %.3f  vs  E[Q1] = %.3f（真值 1.0）' % (m, q1.mean()))
assert m < q1.mean() - 0.3
print('min 有意压低估计（低估不致命，高估会被 Actor 放大 → 低估比高估安全）')
print('配合延迟更新（Critic 先学准）与目标平滑噪声，TD3 把 DDPG 的脆弱性补掉')

E[min(Q1,Q2)] = 0.438  vs  E[Q1] = 1.002（真值 1.0）
min 有意压低估计（低估不致命，高估会被 Actor 放大 → 低估比高估安全）
配合延迟更新（Critic 先学准）与目标平滑噪声，TD3 把 DDPG 的脆弱性补掉


### 熵正则：SAC 为什么"主动鼓励随机"（🔴 深入）

**直觉问题**：PG 的探索靠策略自带随机性；但训练后期策略变"尖"（贪心），
随机性自然消失 → 探索枯竭。SAC 的做法：**把熵写进目标**，
让"随机性本身"成为被优化的东西：

$$J(\theta) = \mathbb{E}\big[\underbrace{r}_{\text{回报}} + \underbrace{\alpha\, H(\pi(\cdot|s))}_{\text{熵奖励}}\big]$$

**为什么这有用（三点）**：
1. **持续探索**：熵大 = 动作分布平 = 不容易早停在一个局部最优
2. **鲁棒性**：学过"多条路都能到"，比只学一条死路的策略抗扰动
3. **与玻尔兹曼策略的关系（推导）**：对固定 $\alpha$，最优策略正是
   $\pi^*(a|s) \propto \exp\big(Q(s,a)/\alpha\big)$——**Q 越大概率越高，
   α 控制"软硬"**（α→0 变 greedy，α→∞ 变均匀）

**温度自适应（对偶上升）**：目标熵 $\bar{H}$ 给定后，
$$\min_\alpha\ \mathbb{E}\big[-\alpha\log\pi - \alpha\bar{H}\big]
\;\Rightarrow\; \alpha \leftarrow \alpha + \beta(\bar{H} - H)$$
熵低于目标 → 升 α（更随机）；高于 → 降 α。**α 不再手调**（本篇 code cell
演示了 α 收敛到正值、熵贴住目标）。

> 一句话：**SAC = 把"多探索"变成数学目标；α 自动调节探索强度，**
> **这是它比 TD3 更稳、更常用的原因。**

---


## 6. SAC：熵正则化的最大熵强化学习（🔴 现代默认）

### 从"最大回报"到"最大回报+熵"

DDPG/TD3 只优化 $J = \mathbb{E}[\sum_t r_t]$。**SAC（Soft Actor-Critic，Haarnoja 2018）** 优化：

$$J = \mathbb{E}_{\pi}\Big[\sum_t r_t + \alpha\, \mathcal{H}\big(\pi(\cdot|s_t)\big)\Big]$$

其中熵 $\mathcal{H}(\pi) = -\mathbb{E}_a[\log \pi(a|s)]$ 衡量随机性。**通俗理解**：
在"拿更多奖励"和"保持动作多样"之间做平衡——奖励差不多时，SAC 会偏向更随机的动作（留余地、抗干扰）。

### 软贝尔曼与最优策略闭式解（推导）

定义**软 Q 迭代**：$Q(s,a) \leftarrow r + \gamma\,\mathbb{E}_{s'}\big[V(s')\big]$，其中
$V(s') = \mathbb{E}_{a'\sim\pi}\big[Q(s',a') - \alpha\log\pi(a'|s')\big]$。

对固定 $Q$ 求最优策略（熵项让目标对 $\pi$ 凹）：
$$\pi^*(a|s) = \frac{1}{Z(s)}\exp\Big(\frac{Q(s,a)}{\alpha}\Big)$$
即**玻尔兹曼（softmax）策略**：Q 高的动作概率指数级更大；$\alpha$ 大 → 分布平（多探索）。

### 温度 α 的自适应（对偶梯度）

α 不是拍脑袋定的：给定目标熵 $\bar{H}$（如 -dim(a)），用**对偶上升**更新：
$$\alpha \leftarrow \alpha - \beta_\alpha \big(\bar{H} - \mathbb{E}[-\log\pi]\big)$$
熵高于目标 → 降 α（收紧探索）；熵低于目标 → 升 α（放宽探索）。

In [5]:
# ========== SAC：策略随 α 的单调性 + 温度自适应（数值） ==========
import numpy as np
# 固定三个动作的 Q 值，看不同 α 下玻尔兹曼策略的熵
logits = np.array([2.0, 1.0, 0.5])
def boltzmann(alpha):
    p = np.exp(logits / alpha); return p / p.sum()
def entropy(p):
    return -np.sum(p * np.log(p))

alphas = np.linspace(0.2, 3.0, 30)
hs = [entropy(boltzmann(a_)) for a_ in alphas]
print('熵随 α 单调增 ✓：α=0.2 → H=%.3f（贪婪）；α=3.0 → H=%.3f（接近均匀）' % (hs[0], hs[-1]))
assert hs[0] < hs[-1]

# 温度自适应（对偶上升）：熵低于目标 → 升 α（更随机）；熵高于目标 → 降 α（更贪婪）
H_target = 0.9; alpha = 0.5; beta_alpha = 0.02
for _ in range(3000):
    H = entropy(boltzmann(alpha))
    alpha = alpha + beta_alpha * (H_target - H)     # 对偶上升：α ← α + β(Ĥ − H)
print('自适应 α 收敛到 %.3f（正值），策略熵 %.3f ≈ 目标 %.2f ✓' % (alpha, H, H_target))
assert abs(H - H_target) < 0.02
print('一句话：SAC = 策略梯度 + 熵正则 + 自动温度 —— 现代连续控制的默认解')

熵随 α 单调增 ✓：α=0.2 → H=0.045（贪婪）；α=3.0 → H=1.076（接近均匀）
自适应 α 收敛到 0.983（正值），策略熵 0.900 ≈ 目标 0.90 ✓
一句话：SAC = 策略梯度 + 熵正则 + 自动温度 —— 现代连续控制的默认解


### 7.1 DDPG / TD3 / SAC 选型精讲（🔴 背诵级）

**本节导读**：三个算法一字排开：DDPG（元祖）、TD3（加固）、SAC
（熵正则）。这一节给"面试半分钟选型"的完整话术。

**1. 三者本质（一句话各）**：
- DDPG：确定性 PG + DQN 双网络（离线样本效率高、但脆）
- TD3：DDPG + 三手术（稳、快、默认替代 DDPG）
- SAC：随机策略 + 最大熵（探索自动、最稳、通常最优）

**2. 逐项对比表（背）**：
| | DDPG | TD3 | SAC |
|---|---|---|---|
| 策略 | 确定性 | 确定性 | 随机（高斯） |
| 探索 | OU 噪声 | 高斯噪声 | 熵内置（α 自适应） |
| 高估 | 高 | min 双Q 压 | min 双Q 压 |
| 目标网络 | 软更新 | 软+延迟 | 软更新 |
| 温度 | — | — | α 自动调 |
| 超参敏感 | 中 | 低 | 低 |
| 默认推荐 | 不用 | 需要确定性 | 通用首选 |

**3. 选型逻辑（决策树）**：
- 环境奖励稀疏 / 需要稳健探索 → **SAC**（熵让策略"广撒网"）
- 动作得是确定输出（部署受限：不能采样）→ **TD3**
- 要与已有 DDPG 代码兼容 / 教学演示 → **DDPG**（了解）+ TD3 升级
- 大状态空间 RLHF 场景？→ 都不是：LLM 用 PPO/GRPO（06 篇）

**4. 为什么 SAC 常赢（直觉）**：熵 = 内置"永不停止的探索"——策略保持
随机直到熵不再带来奖励；α 自适应让"多随机"自动调节（本篇 §6 实验）。
TD3 的确定性、目标平滑是对"脆"的补丁；SAC 从目标函数层面就抗脆。

**5. 易错点**：
- SAC 的"随机"在**执行**时也可采 μ（测试期取均值=确定性，两不误）
- 三者都是 off-policy（回放可用）——别和 PPO(on-policy) 混

**6. 面试问法**："连续控制选哪个？" → "默认 SAC（熵正则自动探索最稳），
要求确定性输出或预算紧选 TD3；DDPG 是 TD3 的历史原型。"

---


## 7. 三兄弟横向对比与选型（🔴 背诵级）

| 算法 | 关键机制 | 探索方式 | 优点 | 缺点 | 一句话选型 |
|---|---|---|---|---|---|
| **DDPG** | DPG 定理 + 四网络 | OU 噪声 | 简单、方差小 | Q 高估、易脆 | 入门/面试推导 |
| **TD3** | 双 Q min + 延迟 + 平滑 | OU/高斯 | 稳定、SOTA 基座 | 实现细节多 | 需要稳的工程 |
| **SAC** | 熵正则 + 温度自适应 | 策略自带随机性 | 探索充分、最鲁棒 | 计算稍贵、调 α | 现代默认选项 |

> 面试金句：**「SAC 是最大熵的 AC；TD3 是防高估的 DDPG；三者的共同点是 Actor 输出连续动作、
> Critic 评价 (s,a)。」

## 8. 数字敏感度与易错点（扩展背诵）

- **τ（软更新系数）**：0.005 量级，太大→目标漂移快、不稳；太小→收敛慢
- **TD3 延迟更新**：Actor 每 **2** 步更新 1 次（标准实现）
- **目标平滑噪声**：$\sigma \approx 0.2$，截断 $c \approx 0.5$（相对动作范围）
- **SAC 目标熵**：连续高斯动作常用 $\bar{H} = -\dim(\mathcal{A})$（如 2 关节 → -2）
- **DPG vs 随机 PG**：DPG 无动作采样（方差小但需 Q 可微）；随机 PG 天然带探索
- 易错：**DDPG 的目标动作来自目标 Actor**（不是在线 Actor）；软更新要**同时**更新 4 个网络中的 2 个目标网络

## 9. 面试速答（30 秒背诵版）

1. **Q：连续动作为什么不能用 DQN？** A：argmax 需要枚举动作，连续空间无穷多。
2. **Q：DPG 定理公式？** A：$\nabla_\theta J = \mathbb{E}_s[\nabla_a Q(s,a)|_{a=\mu_\theta(s)}\cdot\nabla_\theta\mu_\theta(s)]$
3. **Q：DDPG 为什么用软更新？** A：目标值变化太快的 TD 训练会发散；τ 小 → 目标慢变 → 稳定
4. **Q：TD3 三改进？** A：双 Q 取 min（防高估）、延迟更新（Critic 先学准）、目标平滑（防钻尖峰）
5. **Q：SAC 和 TD3 区别？** A：SAC 显式优化熵（自带探索+自动调温），TD3 靠噪声探索
6. **Q：OU 噪声为什么比白噪声适合连续控制？** A：时间相关 → 动作平滑，机械臂不抖

## 10. 知识链自测

- [ ] 能手推：DPG 定理四步推导；SAC 玻尔兹曼最优策略
- [ ] 能对比：DQN vs DDPG 的目标值写法；TD3 vs DDPG 的三处改动
- [ ] 能默写：软更新公式、TD3 目标 y、SAC 温度对偶更新
- [ ] 下一篇（09-MARL）预告：单智能体的 Actor 学会之后，**多个 Actor 同时学**会出什么新问题？